In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging as langsmith_logging

# API KEY 정보로드
load_dotenv()
# Langsmith 로깅 설정
langsmith_logging.langsmith("RAG-EXAMPLE-01")


LangSmith 추적을 시작합니다.
[프로젝트명]
RAG-EXAMPLE-01


In [2]:
from langchain.document_loaders import PDFPlumberLoader
from langchain_upstage.embeddings import UpstageEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.prompts import load_prompt
from langchain_core.runnables import RunnablePassthrough
from langchain_cohere import CohereRerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

from example.embeddings import DebugUpstageAsymmetricEmbeddings
from example.vector_store import get_or_create_vector_store, debug_embedding_process, debug_retrieval_process

# 문서 파싱
loader = PDFPlumberLoader("./data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()
print(f"✅파싱된 문서의 수: {len(docs)}")

# 문서 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter.split_documents(docs)
print(f"✅분할된 문서의 수: {len(split_docs)}")

# 임베딩
embeddings = DebugUpstageAsymmetricEmbeddings()

# 벡터스토어 저장
vector_store = get_or_create_vector_store(
    documents=split_docs,
    embedding=embeddings,
)

✅파싱된 문서의 수: 23
✅분할된 문서의 수: 43
새로운 벡터스토어를 생성합니다: ./data/chroma
디렉토리 생성: ./data/chroma


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


[DEBUG] embed_documents() 호출 → 모델: solar-embedding-1-large-passage


In [3]:
# ChatOpenAI 사용
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# 프롬프트 로드
prompt = load_prompt("prompts/rag-prompts.yaml")


# 기존 함수도 유지 (호환성을 위해)
def format_docs(docs) -> str:
    """검색된 문서를 프롬프트용 문자열로 직렬화한다."""
    return "\n\n".join(
        f"<document><content>{doc.page_content}</content><page>{doc.metadata.get('page', 'Unknown')}</page><source>{doc.metadata.get('source', 'Unknown')}</source></document>"
        for doc in docs
    )

In [4]:
query_embedder = DebugUpstageAsymmetricEmbeddings()
db = get_or_create_vector_store(
    embedding=query_embedder,
)

retriever = db.as_retriever(search_kwargs={"k": 10})
compressor = CohereRerank(model="rerank-multilingual-v3.0")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever,
)

chain = (
    {"context": compression_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


기존 벡터스토어를 로드했습니다: ./data/chroma


In [5]:
output = chain.invoke("삼성전자에서 개발한 생성형AI에 대해 설명해줘")
print(output)

[DEBUG] embed_query() 호출 → 모델: solar-embedding-1-large-query


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


삼성전자가 개발한 생성형 AI는 '삼성 가우스'라는 이름으로, 온디바이스에서 작동할 수 있는 모델로 구성되어 있습니다. 이 AI는 언어, 코드, 이미지의 세 가지 모델로 나뉘며, 각각의 모델은 다음과 같은 기능을 제공합니다:

1. **언어 모델**: 텍스트 생성, 문서 요약, 번역 등의 작업을 지원합니다.
2. **코드 모델**: AI 코딩 어시스턴트인 '코드아이(code.i)'를 통해 대화형 인터페이스로 소프트웨어 개발에 최적화된 서비스를 제공합니다.
3. **이미지 모델**: 창의적인 이미지를 생성하고 기존 이미지를 수정할 수 있으며, 저해상도 이미지를 고해상도로 변환하는 기능도 지원합니다.

삼성 가우스는 안전한 데이터로 학습되었으며, 온디바이스에서 작동하기 때문에 사용자 정보 유출의 위험이 없습니다. 삼성전자는 이 AI 기술을 다양한 제품에 단계적으로 탑재할 계획입니다.

출처 (Sources)
- [1] SPRI_AI_Brief_2023년12월호_F.pdf, Page 12
- [2] SPRI_AI_Brief_2023년12월호_F.pdf, Page 1
